## Combine region files -> countryscale
## Map region -> to netwerkschakels

In [3]:

# --- CELL 1: Aggregation, Dissolve, and Area Overlay ---

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd

TARGET_CRS = "EPSG:28992"
region_list = [
    "Groningen en NO-Drenthe", "Noord-Westelijke Delta", "Overijsselse Vecht",
    "Limburg", "Vallei en Veluwe", "Achterhoek", "Brabantse Delta",
    "Friesland", "ARK-NZK", "Noord-Brabant Oost", "Rivierenland",
    "Scheldestromen", "Zuiderzeeland"
]
keep_cols = [
    "NETWERKSCH_HWN", "total_length", "flooded_length", "bridge_length_sum",
    "tunnel_length_sum", "total_damage", "F_EV2_ma_max", "geometry"
]
additive_cols = ["flooded_length", "bridge_length_sum", "tunnel_length_sum", "total_damage"]
numeric_cols = ["total_length"] + additive_cols + ["F_EV2_ma_max"]

gdfs = []
for region in region_list:
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    gpkg_path = root_dir / "Aggregated_schakels.gpkg"
    if not gpkg_path.exists():
        warnings.warn(f"Missing file, skipping: {gpkg_path}")
        continue
    try:
        gdf = gpd.read_file(gpkg_path)
    except Exception as e:
        warnings.warn(f"Failed to read {gpkg_path}: {e}")
        continue
    if "NETWERKSCH_HWN" not in gdf.columns:
        warnings.warn(f"'NETWERKSCH_HWN' column missing in {gpkg_path}, skipping")
        continue
    if gdf.crs is None:
        gdf.set_crs(TARGET_CRS, inplace=True)
    if gdf.crs.to_string() != TARGET_CRS:
        gdf = gdf.to_crs(TARGET_CRS)
    for col in additive_cols:
        if col not in gdf.columns:
            gdf[col] = 0.0
    if "total_length" not in gdf.columns:
        gdf["total_length"] = np.nan
    if "F_EV2_ma_max" not in gdf.columns:
        gdf["F_EV2_ma_max"] = np.nan
    for col in numeric_cols:
        gdf[col] = pd.to_numeric(gdf[col], errors="coerce")
    for col in keep_cols:
        if col not in gdf.columns:
            gdf[col] = 0.0 if col in additive_cols else (np.nan if col != "geometry" else gdf.geometry)
    gdfs.append(gdf[keep_cols])

if not gdfs:
    raise SystemExit("No GeoPackages found to merge.")

df = pd.concat(gdfs, ignore_index=True)

# quick check: show groups with more than one row (helps debug why og_ may contain single values)
vc = df["NETWERKSCH_HWN"].value_counts()
print("NETWERKSCH_HWN counts (top 10):")
print(vc.head(10))
print("Number of NETWERKSCH_HWN with >1 row:", (vc > 1).sum())

# create og-aggregates: exclude geometry and the key
og_cols = [c for c in keep_cols if c not in ("NETWERKSCH_HWN", "geometry")]
# keep all original values (dropna) as comma-separated strings per group
og_df = (
    df.groupby("NETWERKSCH_HWN", as_index=False)[og_cols]
      .agg(lambda s: ",".join(s.dropna().astype(str)))
)
# prefix aggregated columns with og_
og_df = og_df.rename(columns={c: f"og_{c}" for c in og_cols})

# --- build aggregated geometry and numeric summaries ---
g = df.groupby("NETWERKSCH_HWN", dropna=False)

def pick_total_length(s: pd.Series) -> float:
    vals = pd.to_numeric(s, errors="coerce").dropna().to_numpy()
    if len(vals) == 0:
        return np.nan
    v0 = float(vals[0])
    if not np.allclose(vals, v0, rtol=1e-6, atol=1e-6):
        warnings.warn(f"Inconsistent total_length for NETWERKSCH_HWN='{s.name}': {vals.tolist()}")
    return v0

geom_df = df[["NETWERKSCH_HWN", "geometry"]].dissolve(by="NETWERKSCH_HWN")
geom_df["NETWERKSCH_HWN"] = geom_df.index  # keep dissolve key as column

# Aggregated numeric values
sums_df = g[additive_cols].sum(min_count=1)
max_df = g["F_EV2_ma_max"].max().rename("F_EV2_ma_max")
totlen_df = g["total_length"].apply(pick_total_length).to_frame(name="total_length")

# Join aggregated values to geometry
merged = geom_df.join([totlen_df, sums_df, max_df])

# Reset index and ensure NETWERKSCH_HWN is a column (not index)
merged = merged.reset_index(drop=True)
merged = merged[["NETWERKSCH_HWN"] + [c for c in merged.columns if c != "NETWERKSCH_HWN"]]

# Merge the og_ aggregates (do NOT overwrite geometry)
# og_df has no geometry column; safe to merge
merged = merged.merge(og_df, on="NETWERKSCH_HWN", how="left")

# remove duplicated columns if any
merged = merged.loc[:, ~merged.columns.duplicated()]

# derive type/NET and convert to GeoDataFrame
_tmp = merged["NETWERKSCH_HWN"].astype("string")
merged["type"] = _tmp.str.extract(r"-(L|R|M)$")[0]
merged["NET"] = _tmp.str.replace(r"-(L|R|M)$", "", regex=True)
del _tmp

merged = gpd.GeoDataFrame(merged, geometry="geometry", crs=TARGET_CRS)

# computed fields
merged["fraction_flooded"] = np.where(
    (merged["total_length"] > 0) & merged["total_length"].notna(),
    merged["flooded_length"] / merged["total_length"],
    np.nan,
)
merged["dam_m"] = np.where(
    (merged["flooded_length"] > 0) & merged["flooded_length"].notna(),
    merged["total_damage"] / merged["flooded_length"],
    np.nan,
)

out_dir = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs")
out_dir.mkdir(parents=True, exist_ok=True)

# --- Area overlay ---
def add_area_overlay(gdf: gpd.GeoDataFrame, area_path: Path, target_crs: str, key_col: str) -> gpd.GeoDataFrame:
    import fiona
    if not area_path.exists():
        warnings.warn(f"Area file not found: {area_path}. Skipping area overlay.")
        return gdf.copy()
    layers = fiona.listlayers(str(area_path))
    layer = layers[0]
    area = gpd.read_file(area_path, layer=layer)
    if area.crs is None:
        area.set_crs(target_crs, inplace=True)
    if area.crs.to_string() != target_crs:
        area = area.to_crs(target_crs)
    candidate_labels = ["name", "id"]
    label_col = next((c for c in candidate_labels if c in area.columns), None)
    new_col = area_path.stem
    left = gdf[[key_col, "geometry"]].copy()
    right_cols = ["geometry"] + ([label_col] if label_col else [])
    sj = gpd.sjoin(left, area[right_cols], how="left", predicate="intersects")
    has_overlap = (
        sj.groupby(key_col)["index_right"]
          .apply(lambda s: s.notna().any())
          .rename(new_col)
          .reset_index()
    )
    out = gdf.merge(has_overlap, on=key_col, how="left")
    out[new_col] = out[new_col].fillna(False)
    if label_col:
        labels = (
            sj.groupby(key_col)[label_col]
              .apply(lambda s: ";".join(sorted({str(v) for v in s.dropna()})) if s.notna().any() else None)
              .rename(f"{new_col}_{label_col}")
              .reset_index()
        )
        out = out.merge(labels, on=key_col, how="left")
    return out

area_gpkg = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Areas.gpkg")
merged = add_area_overlay(merged, area_gpkg, TARGET_CRS, key_col="NETWERKSCH_HWN")


NETWERKSCH_HWN counts (top 10):
027-0090-L    3
050-0070-R    3
050-0070-L    3
027-0090-R    3
002-0080-R    2
050-0100-L    2
050-0100-M    2
050-0100-R    2
050-0110-L    2
050-0110-R    2
Name: NETWERKSCH_HWN, dtype: int64
Number of NETWERKSCH_HWN with >1 row: 102


In [2]:
from shapely.ops import unary_union

def apply_type_edited_and_merge(gdf, enable_type_edited=True):
    if not enable_type_edited:
        gdf["type_edited"] = gdf["type"]
        gdf["scakel_ed"] = (gdf["NET"].fillna("") + "_" + gdf["type_edited"].fillna("")).str.strip("_")
        return gdf

    idx_longest = gdf.groupby("NET")["total_length"].idxmax()
    longest_type_by_net = gdf.loc[idx_longest, ["NET", "type"]].set_index("NET")["type"]
    has_small_seg = gdf.groupby("NET")["total_length"].transform(lambda s: (s < 4000).any())
    gdf["type_edited"] = np.where(
        (has_small_seg) & (gdf["total_length"] < 4000),
        gdf["NET"].map(longest_type_by_net),
        gdf["type"],
    )
    gdf["scakel_ed"] = (gdf["NET"].fillna("") + "_" + gdf["type_edited"].fillna("")).str.strip("_")

    agg_dict = {
        "total_length": "sum",
        "flooded_length": "sum",
        "bridge_length_sum": "sum",
        "tunnel_length_sum": "sum",
        "total_damage": "sum",
        "F_EV2_ma_max": "max",
        "geometry": lambda x: unary_union(x),
    }

    def pick_from_largest_total_length(series):
        idx = gdf.loc[series.index, "total_length"].idxmax()
        return gdf.loc[idx, series.name] if series.name in gdf.columns else series.iloc[0]

    for col in gdf.columns:
        if col not in agg_dict and col not in ["NET", "type_edited", "scakel_ed"]:
            agg_dict[col] = pick_from_largest_total_length

    gdf_agg = gdf.groupby(["NET", "type_edited", "scakel_ed"], as_index=False).agg(agg_dict)
    return gdf_agg

In [4]:
# --- CELL 2: Optional type_edited logic and final export ---


# After apply_type_edited_and_merge
#merged = apply_type_edited_and_merge(merged, enable_type_edited=True)

# Convert to GeoDataFrame (ensure geometry column is present)
merged = gpd.GeoDataFrame(merged, geometry="geometry", crs=TARGET_CRS)

edited_base = "Aggregated_schakels_merged_with_area_edited_ver04"
gpkg_out = out_dir / f"{edited_base}.gpkg"
layer_out = edited_base.replace(" ", "_")
merged.to_file(gpkg_out, driver="GPKG", layer=layer_out)
merged.to_parquet(out_dir / f"{edited_base}.parquet", index=False)

shp_ready = merged.copy()
for col in shp_ready.select_dtypes(include=["bool"]).columns:
    shp_ready[col] = shp_ready[col].astype("uint8")
shp_out = out_dir / f"{edited_base}.shp"
shp_ready.to_file(shp_out, driver="ESRI Shapefile")

print(f"Wrote edited outputs: {gpkg_out} (layer={layer_out}), {edited_base}.parquet, and {shp_out}")

C:\Users\gunaratn\AppData\Local\Temp\ipykernel_13260\423801937.py:20: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  shp_ready.to_file(shp_out, driver="ESRI Shapefile")


Wrote edited outputs: P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04.gpkg (layer=Aggregated_schakels_merged_with_area_edited_ver04), Aggregated_schakels_merged_with_area_edited_ver04.parquet, and P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04.shp


In [3]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import fiona  # <-- Add this line

region_list = [
    "ARK-NZK","Vallei en Veluwe","Achterhoek", "Brabantse Delta","Friesland",
    "Groningen en NO-Drenthe","Limburg","Noord-Brabant Oost","Noord-Westelijke Delta",
    "Rivierenland","Scheldestromen","Zuiderzeeland","Overijsselse Vecht"
]

all_roads = []

for region in region_list:
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    road_path = root_dir / "allowed_lowerlying_flooded_dissolved.gpkg"
    if not road_path.exists():
        print(f"File not found for region {region}: {road_path}. Skipping.")
        continue
    # List all layers in the GPKG
    layers = fiona.listlayers(str(road_path))  # <-- Fix here
    for layer in layers:
        road = gpd.read_file(road_path, layer=layer)
        road['region'] = region
        road['source_layer'] = layer  # Optionally keep track of the layer
        all_roads.append(road)

if not all_roads:
    raise SystemExit("No data found in any region.")

merged_roads = pd.concat(all_roads, ignore_index=True)
# Convert to GeoDataFrame if not already
if not isinstance(merged_roads, gpd.GeoDataFrame):
    merged_roads = gpd.GeoDataFrame(merged_roads, geometry='geometry')

out_dir = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs")
out_dir.mkdir(parents=True, exist_ok=True)
merged_roads.to_parquet(out_dir / "allowed_lowerlying_flooded_dissolved.parquet", index=False)
print(f"Wrote merged parquet to {out_dir / 'allowed_lowerlying_flooded_dissolved.parquet'}")

File not found for region Vallei en Veluwe: P:\bovenregionale-stresstest-hwn\Analysis\Vallei en Veluwe\Outputs\allowed_lowerlying_flooded_dissolved.gpkg. Skipping.
File not found for region Achterhoek: P:\bovenregionale-stresstest-hwn\Analysis\Achterhoek\Outputs\allowed_lowerlying_flooded_dissolved.gpkg. Skipping.
File not found for region Friesland: P:\bovenregionale-stresstest-hwn\Analysis\Friesland\Outputs\allowed_lowerlying_flooded_dissolved.gpkg. Skipping.
File not found for region Groningen en NO-Drenthe: P:\bovenregionale-stresstest-hwn\Analysis\Groningen en NO-Drenthe\Outputs\allowed_lowerlying_flooded_dissolved.gpkg. Skipping.
File not found for region Rivierenland: P:\bovenregionale-stresstest-hwn\Analysis\Rivierenland\Outputs\allowed_lowerlying_flooded_dissolved.gpkg. Skipping.
File not found for region Scheldestromen: P:\bovenregionale-stresstest-hwn\Analysis\Scheldestromen\Outputs\allowed_lowerlying_flooded_dissolved.gpkg. Skipping.
File not found for region Zuiderzeeland:

In [2]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import fiona

region_list = ["ARK-NZK","Vallei en Veluwe",
               "Achterhoek", "Brabantse Delta","Friesland",
               "Groningen en NO-Drenthe","Limburg",
               "Noord-Brabant Oost","Noord-Westelijke Delta",
               "Rivierenland","Scheldestromen","Zuiderzeeland",
               "Overijsselse Vecht"
               ]

all_roads = []

for region in region_list:
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    road_path = root_dir / "allowed_lowerlying_flooded_dissolved.gpkg"
    if not road_path.exists():
        print(f"File not found for region {region}: {road_path}. Skipping.")
        continue
    # List all layers in the GPKG
    layers = fiona.listlayers(str(road_path))
    for layer in layers:
        road = gpd.read_file(road_path, layer=layer)
        road['region'] = region
        road['source_layer'] = layer  # Optionally keep track of the layer
        all_roads.append(road)

if not all_roads:
    raise SystemExit("No data found in any region.")

merged_roads = pd.concat(all_roads, ignore_index=True)
# Convert to GeoDataFrame if not already
if not isinstance(merged_roads, gpd.GeoDataFrame):
    merged_roads = gpd.GeoDataFrame(merged_roads, geometry='geometry')

out_dir = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs")
out_dir.mkdir(parents=True, exist_ok=True)

# Export as Parquet
merged_roads.to_parquet(out_dir / "allowed_lowerlying_flooded_dissolved.parquet", index=False)
print(f"Wrote merged parquet to {out_dir / 'allowed_lowerlying_flooded_dissolved.parquet'}")

# Prepare and export as Shapefile
shp_ready = merged_roads.copy()
# Convert boolean columns to uint8 for Shapefile compatibility
for col in shp_ready.select_dtypes(include=["bool"]).columns:
    shp_ready[col] = shp_ready[col].astype("uint8")

# Truncate column names to 10 characters (Shapefile limitation)
col_mapping = {}
for col in shp_ready.columns:
    if len(col) > 10:
        col_mapping[col] = col[:10]
        
if col_mapping:
    shp_ready = shp_ready.rename(columns=col_mapping)
    print(f"Truncated {len(col_mapping)} column names for Shapefile compatibility")

shp_out = out_dir / "allowed_lowerlying_flooded_dissolved.shp"
shp_ready.to_file(shp_out, driver="ESRI Shapefile")
print(f"Wrote merged shapefile to {shp_out}")

File not found for region Vallei en Veluwe: P:\bovenregionale-stresstest-hwn\Analysis\Vallei en Veluwe\Outputs\allowed_lowerlying_flooded_dissolved.gpkg. Skipping.
File not found for region Achterhoek: P:\bovenregionale-stresstest-hwn\Analysis\Achterhoek\Outputs\allowed_lowerlying_flooded_dissolved.gpkg. Skipping.
File not found for region Brabantse Delta: P:\bovenregionale-stresstest-hwn\Analysis\Brabantse Delta\Outputs\allowed_lowerlying_flooded_dissolved.gpkg. Skipping.
File not found for region Friesland: P:\bovenregionale-stresstest-hwn\Analysis\Friesland\Outputs\allowed_lowerlying_flooded_dissolved.gpkg. Skipping.
File not found for region Groningen en NO-Drenthe: P:\bovenregionale-stresstest-hwn\Analysis\Groningen en NO-Drenthe\Outputs\allowed_lowerlying_flooded_dissolved.gpkg. Skipping.
File not found for region Limburg: P:\bovenregionale-stresstest-hwn\Analysis\Limburg\Outputs\allowed_lowerlying_flooded_dissolved.gpkg. Skipping.
File not found for region Noord-Brabant Oost: P: